STEP 1: Install Libraries

In [1]:
!pip install transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00


STEP 2: Load Model

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

STEP 3: Create Legal Dataset

In [3]:
from datasets import Dataset

data = [
    {"question": "What is a contract?", "answer": "A contract is a legally enforceable agreement between two or more parties creating mutual obligations."},
    {"question": "What is breach of contract?", "answer": "A breach occurs when a party fails to perform obligations under the agreement."},
    {"question": "What is negligence?", "answer": "Negligence is the failure to exercise reasonable care resulting in harm."},
    {"question": "What is liability?", "answer": "Liability refers to legal responsibility for one's actions."},
    {"question": "What is indemnity?", "answer": "Indemnity is a contractual obligation to compensate for loss or damage."},
    {"question": "What is tort law?", "answer": "Tort law governs civil wrongs that cause harm."},
    {"question": "What is consideration?", "answer": "Consideration is something of value exchanged in a contract."},
    {"question": "What is damages?", "answer": "Damages refer to monetary compensation for loss or injury."},
    {"question": "What is arbitration?", "answer": "Arbitration is a dispute resolution method involving a neutral third party."},
    {"question": "What is jurisdiction?", "answer": "Jurisdiction is the legal authority of a court to hear a case."}
]

dataset = Dataset.from_list(data)

STEP 4: Format Data

In [4]:
def format_data(example):
    return {
        "text": f"<|user|>\n{example['question']}\n<|assistant|>\n{example['answer']}"
    }

dataset = dataset.map(format_data)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

STEP 5: Apply LoRA

In [5]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # ✅ correct
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

STEP 6: Tokenization

In [6]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

STEP 7: Training

In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=10,
    logging_steps=1,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Step,Training Loss
1,17.242386
2,16.158684
3,14.595951
4,13.174326
5,12.005901
6,9.975153
7,7.662029
8,6.689528
9,5.925692
10,4.915998


TrainOutput(global_step=50, training_loss=2.9289889922738075, metrics={'train_runtime': 53.5054, 'train_samples_per_second': 1.869, 'train_steps_per_second': 0.934, 'total_flos': 159593162342400.0, 'train_loss': 2.9289889922738075, 'epoch': 10.0})

STEP 8: Generate Answer

In [8]:
def generate_answer(question):
    prompt = f"<|user|>\n{question}\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=150,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

STEP 9: Clean Output

In [9]:
def clean_output(text):
    return text.split("<|assistant|>")[-1].strip()

STEP 10: Test Model

In [10]:
response = generate_answer("What is a contract?")
print(clean_output(response))

A contract is an agreement between two or more parties that sets forth the terms and conditions of their relationship, including responsibilities, expectations, obligations, and rights.


STEP 11: BEFORE vs AFTER

In [11]:
base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

def base_generate(question):
    prompt = f"<|user|>\n{question}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    outputs = base_model.generate(**inputs, max_length=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("BEFORE:\n", base_generate("What is a contract?"))
print("\nAFTER:\n", clean_output(generate_answer("What is a contract?")))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BEFORE:
 <|user|>
What is a contract?
<|assistant|>
A contract is a legal agreement between two or more parties that sets out the terms and conditions of a transaction or relationship. It outlines the rights and obligations of both parties, including the terms of payment, delivery, and performance. Contracts can be written or oral, and they can be signed by both parties or witnessed by a third party.
In the context of the text

AFTER:
 A contract is an agreement between two or more parties that sets forth the terms and conditions under which one party will perform its obligations in exchange for payment from another.
